# Notebook 1 — Strands Agents Fundamentals

This classroom-ready notebook introduces the core Strands agent concepts before moving into multi-agent systems.

## What this notebook covers

By the end of this notebook, learners should understand:

1. What an agent loop is
2. How a brain-only agent works
3. How prompts shape agent behavior
4. How tools extend an agent
5. How custom tools are created with `@tool`
6. How a brain + tool + memory agent works
7. How structured output can make agent responses machine-readable
8. How conversation management protects the context window
9. How session management can persist state across invocations
10. How retry strategies help with rate limits and transient model failures


## Install packages

Run the below cell only if the packages are not already available in the notebook kernel.

In [ ]:
# Uncomment if needed
!pip install -q strands-agents strands-agents-tools boto3 pandas pydantic==2.9

## Imports

The below code imports the main packages used in this notebook.

In [ ]:
import boto3
import pandas as pd
from datetime import datetime
from typing import Optional

from pydantic import BaseModel, Field

from strands import Agent, tool
from strands.models import BedrockModel
from strands import ModelRetryStrategy
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands.session.file_session_manager import FileSessionManager

print("Imports completed")


## Bedrock configuration

The below code configures the Bedrock model used by all agents in this notebook.

Change `MODEL_ID` only if your classroom environment uses a different model.

In [ ]:
AWS_REGION = boto3.session.Session().region_name or "us-east-1"

# Common classroom-friendly models:
# - amazon.nova-lite-v1:0
# - amazon.nova-micro-v1:0
# - amazon.nova-pro-v1:0

MODEL_ID = "amazon.nova-lite-v1:0"

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.2,
)

print("Region:", AWS_REGION)
print("Model:", MODEL_ID)


# Part A — Agent loop mental model

A normal LLM mostly generates text.

An agent can take actions because it runs inside a loop.

## Agent loop

1. User gives a request
2. Agent sends request + context to the model
3. Model decides whether it can answer directly or needs a tool
4. If tool is needed, the tool runs
5. Tool result goes back into the agent context
6. Model reasons again
7. Loop continues until the model gives a final response

This is why agents can solve multi-step tasks instead of only answering from memory.

## Agent with brain only

This first agent has:

| Component | Present? |
|---|---:|
| Brain / LLM | Yes |
| Prompt | Yes |
| Tool | No |
| Memory | No |

It can reason and explain, but it cannot call external tools.

In [ ]:
concept_agent = Agent(
    model=bedrock_model,
    system_prompt="""
You are a teaching assistant for a first class on AI agents.

When explaining a concept:
1. Give a one-line definition.
2. Give a simple classroom example.
3. Give one limitation.
Keep the answer brief.
""",
)
concept_agent("Explain what an AI agent is.")



In [ ]:
response = concept_agent("Explain what an AI agent is.")
print(response)

In [ ]:
# The below code creates a small helper function to print agent responses cleanly. we will use this for the rest of the class
def show_response(title, response):
    print("\n")
    print("=" * 80)
    print(title)
    print("=" * 80)
    print(response)
    
show_response("Brain-only Agent Output", response)

# Part B — Prompts

A prompt is not just a question.

In agents, the system prompt defines the agent's role, boundaries, response style, and tool-use behavior.

## Prompt comparison

The below code creates two agents with different system prompts.

Both use the same model, but their behavior should differ because the instructions differ.

In [ ]:
short_answer_agent = Agent(
    model=bedrock_model,
    system_prompt="""
You are a concise technical explainer.
Answer in exactly three bullet points.
""",
)

teacher_agent = Agent(
    model=bedrock_model,
    system_prompt="""
You are a patient classroom instructor.
Explain using a simple analogy and then one practical example.
""",
)



In [ ]:
# Response 1 - Concise
question = "What is the difference between an LLM and an agent?"
response1 = short_answer_agent(question)
show_response("Prompt Style A — Concise", response1)


In [ ]:
# Response 2 - Teacher style
question = "What is the difference between an LLM and an agent?"
response2 = teacher_agent(question)
show_response("Prompt Style B — Teacher Style", response2)

# Part C — Custom tools

Tools allow an agent to do something beyond text generation.

In Strands, simple custom tools can be created using the `@tool` decorator.

Important teaching point:

- The function name helps identify the tool.
- The docstring helps the model understand when to use it.
- Type hints help define the input schema.

## Create simple classroom tools

The below code creates three tools:

1. Current class time
2. Percentage calculator
3. Learner progress message generator

In [ ]:
@tool
def get_current_class_time() -> str:
    """Return the current date and time for the classroom session."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


@tool
def calculate_percentage(score: float, total: float) -> str:
    """Calculate percentage score from score and total marks."""
    if total == 0:
        return "Total cannot be zero."
    percentage = (score / total) * 100
    return f"{percentage:.2f}%"


@tool
def learner_progress_message(learner_name: str, percentage: float) -> str:
    """Create a short progress message based on learner percentage."""
    if percentage >= 80:
        level = "strong progress"
    elif percentage >= 60:
        level = "steady progress"
    else:
        level = "needs more practice"

    return f"{learner_name} is showing {level}. Current score: {percentage:.1f}%."

## Agent with brain + tools

This agent has:

| Component | Present? |
|---|---:|
| Brain / LLM | Yes |
| Prompt | Yes |
| Tool | Yes |
| Memory | No |

The agent decides which tool to use based on the user request.

In [ ]:
classroom_agent = Agent(
    model=bedrock_model,
    tools=[
        get_current_class_time,
        calculate_percentage,
        learner_progress_message,
    ],
    system_prompt="""
You are a classroom operations agent.

Use tools whenever calculation, date/time, or learner progress generation is requested.
Do not guess tool results.
Keep the final answer brief and clear.
""",
)

response = classroom_agent(
    """
Please do these tasks:
1. Tell me the current class time.
2. Calculate the percentage for 42 out of 50.
3. Create a short progress message for Riya using score 84%.
"""
)

show_response("Brain + Tools Agent Output", response)

# Part D — Agent with brain + tools + memory

So far, the agent can reason and use tools, but it does not remember user-specific information unless we give it a memory mechanism.

In this classroom example, memory is implemented using a simple Python dictionary.

This is not production memory. It is a teaching example.

## Create memory tools

The below code creates three tools:

1. Save learner preference
2. Recall learner preference
3. Calculate discounted price

In [ ]:
agent_memory = {}


@tool
def remember_learner_preference(name: str, preference: str) -> str:
    """Save a learner preference in memory."""
    agent_memory[name.lower()] = preference
    return f"Preference saved for {name}: {preference}"


@tool
def recall_learner_preference(name: str) -> str:
    """Recall a learner preference from memory."""
    preference = agent_memory.get(name.lower())
    if preference is None:
        return f"No preference found for {name}."
    return f"{name}'s remembered preference is: {preference}"


@tool
def calculate_discounted_price(original_price: float, discount_percent: float) -> str:
    """Calculate final price after applying a discount percentage."""
    final_price = original_price * (1 - discount_percent / 100)
    return f"Original price: {original_price:.2f}, Discount: {discount_percent:.1f}%, Final price: {final_price:.2f}"

## Create the memory agent

This agent has:

| Component | Present? |
|---|---:|
| Brain / LLM | Yes |
| Prompt | Yes |
| Tool | Yes |
| Memory | Yes |

In [ ]:
memory_agent = Agent(
    model=bedrock_model,
    tools=[
        remember_learner_preference,
        recall_learner_preference,
        calculate_discounted_price,
    ],
    system_prompt="""
You are a learner support agent.

Use memory tools when the user asks you to remember or recall preferences.
Use calculation tools when the user asks for discount or price calculations.
Do not invent preferences that are not saved in memory.
""",
)

## Test memory across turns

Run the next three cells in order.

In [ ]:
response = memory_agent(
    """
My name is Ravi.
Please remember that I usually want a 15% discount.
"""
)

show_response("Turn 1 — Save Memory", response)
print("Current memory dictionary:", agent_memory)

In [ ]:
response = memory_agent(
    """
What discount did I say I prefer?
Also calculate the final price for 3200 using my preferred discount.
My name is Ravi.
"""
)

show_response("Turn 2 — Recall Memory + Use Tool", response)

In [ ]:
response = memory_agent("What discount does Meera prefer?")
show_response("Turn 3 — Missing Memory", response)

# Classroom Activity 1 — Build a store assistant agent

## Task

Create an agent that helps with simple store operations.

## Requirements

The agent should:

1. Calculate bill totals
2. Apply discount percentages
3. Create a short customer message
4. Use tools where calculations are required

## Expected tools

- `calculate_total_price`
- `apply_discount`
- `create_customer_message`

In [ ]:
# Student Activity 1: Build a store assistant agent
# Create tools using @tool
# Create an Agent with those tools
# Test with this prompt:
# "A customer bought 3 units at 1200 each. Apply 10% discount and write a short message."

# Your code goes here

# Classroom Activity 2 — Build a payroll helper agent

## Task

Build a Smart Payroll Assistant Agent for a mid-sized company.

## Requirements

The agent should:

1. Calculate gross pay
2. Calculate deduction amount
3. Calculate net pay
4. Explain the result in simple language

## Expected tools

- `calculate_gross_pay`
- `calculate_deductions`
- `calculate_net_pay`

In [ ]:
# Student Activity 2: Build a payroll helper agent

# Your code goes here

# Classroom Activity 3 — Add memory to an agent

## Task

Extend either Activity 1 or Activity 2 by adding simple memory.

## Example memory behavior

- Remember employee department
- Remember customer discount preference
- Remember learner preferred explanation style

## Expected tools

- one tool to save memory
- one tool to recall memory

In [ ]:
# Student Activity 3: Add memory to an agent

# Your code goes here

# Part E — Structured output

Free-text answers are useful for humans.

Structured output is useful when another system must read the result.

Example use cases:

- Evaluation reports
- Risk classification
- Test case generation
- Ticket creation
- Agent handoff payloads

## Define a structured output schema

The below code defines the expected response structure using Pydantic.

In [ ]:
class ReviewResult(BaseModel):
    summary: str = Field(description="One-sentence summary of the reviewed item")
    risk_level: str = Field(description="Low, Medium, or High")
    missing_information: list[str] = Field(description="Important missing details")
    recommended_next_action: str = Field(description="The next best action")


structured_review_agent = Agent(
    model=bedrock_model,
    system_prompt="""
You are a strict but helpful reviewer.
When reviewing a feature description, identify gaps and classify risk.
""",
)

## Ask for structured output

The result should include a `structured_output` object that follows the schema.

In [ ]:
feature_description = """
Feature: Password reset using email OTP.
The user enters an email address and receives an OTP.
After entering the OTP, the user can create a new password.
"""

result = structured_review_agent(
    feature_description,
    structured_output_model=ReviewResult,
)

print("Raw response:")
print(result)

print("\nStructured output fields:")
print("Summary:", result.structured_output.summary)
print("Risk level:", result.structured_output.risk_level)
print("Missing information:", result.structured_output.missing_information)
print("Recommended next action:", result.structured_output.recommended_next_action)

# Part F — Conversation management

Agent conversations can become long.

Conversation management decides how much prior conversation should be retained in the model context.

For a classroom demo, a sliding window conversation manager is easiest to explain:

- keep the most recent messages
- avoid sending the entire conversation forever
- reduce context window pressure

## Create an agent with a custom conversation window


In [ ]:
conversation_manager = SlidingWindowConversationManager(
    window_size=6,
    should_truncate_results=True,
)

conversation_agent = Agent(
    model=bedrock_model,
    conversation_manager=conversation_manager,
    system_prompt="""
    You are a concise assistant.
    Use the recent conversation context, but do not over-explain.
    """,
)



In [ ]:
response = conversation_agent("My name is Ravi and I am learning agents.")


In [ ]:
response = conversation_agent("I prefer short examples.")


In [ ]:
response = conversation_agent("What do you remember about me from this conversation?")

show_response("Conversation Manager Output", response)

# Part G — Session management

The memory dictionary earlier works only while the notebook kernel is alive.

Session management is different. It is used to persist agent conversation history and state across invocations, and depending on storage, across restarts.

In this classroom notebook, we use a file-based session manager if available.

In production, you may use more durable storage such as S3.

## Create an agent with file session management



In [ ]:
session_manager = FileSessionManager(
    session_id="classroom-single-agent-session",
    storage_dir="./agent_sessions",
)

session_agent = Agent(
    model=bedrock_model,
    session_manager=session_manager,
    system_prompt="""
    You are a session-aware classroom assistant.
    Use the session context when answering follow-up questions.
    """,
)


In [ ]:
response1 = session_agent("My project topic is payroll automation. Remember this in our session.")
response2 = session_agent("What is my project topic?")

show_response("Session Turn 1", response1)
show_response("Session Turn 2", response2)

# Part H — Retry strategy

Model APIs can occasionally fail because of throttling, timeouts, or temporary service issues.

A retry strategy controls how the agent retries model calls.

This is important in production because many may call the model at the same time.

## Configure a retry strategy

The below code creates an agent with retry configuration if `ModelRetryStrategy` is available in the installed SDK version.

In [ ]:
resilient_agent = Agent(
    model=bedrock_model,
    retry_strategy=ModelRetryStrategy(
        max_attempts=3,
        initial_delay=2,
        max_delay=30,
    ),
    system_prompt="""
    You are a reliable classroom assistant.
    Answer clearly and briefly.
    """,
)

response = resilient_agent("Explain why retry strategy is useful for agents.")
show_response("Retry Strategy Agent Output", response)

# Part K — Comparison table

| Example | Brain | Prompt | Tool | Memory | Structured Output | Session | Main learning |
|---|---:|---:|---:|---:|---:|---:|---|
| Brain-only agent | Yes | Yes | No | No | No | No | LLM can reason and respond |
| Prompt comparison | Yes | Yes | No | No | No | No | System prompt changes behavior |
| Tool agent | Yes | Yes | Yes | No | No | No | Agent can perform actions |
| Memory agent | Yes | Yes | Yes | Yes | No | No | Agent can remember small facts using tools |
| Structured output agent | Yes | Yes | Optional | No | Yes | No | Agent output can be machine-readable |
| Conversation manager agent | Yes | Yes | Optional | Conversation context | No | No | Agent controls context window |
| Session agent | Yes | Yes | Optional | State/history | No | Yes | Agent can persist session context |
| Retry strategy agent | Yes | Yes | Optional | No | No | No | Agent can handle transient API failures |

# Wrap-up

In this notebook, learners moved from a basic agent to a more complete single-agent pattern:

**Brain → Prompt → Tool → Memory → Structured Output → Conversation Management → Session Management → Retry Strategy**

This prepares learners for Notebook 2, where the focus shifts from tools to **Skills**.